# Решения: CI и корреляция

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

## Карта эталона

**Фокус:** Неопределённость эффекта и границы корреляции.

Bootstrap показывает диапазон правдоподобных uplift, а корреляции помогают искать связи. Ни один из этих инструментов сам по себе не доказывает причинность.

Эталон разделён на исполняемые секции в том же порядке, что `lesson.ipynb` и `homework.ipynb`. После каждой секции сверяйте не только значение, но и способ вычисления.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def _find(name):
    for path in (
        Path(name),
        Path("../") / name,
        Path("../../data") / name,
        Path("../data") / name,
        Path("../../../data") / name,
    ):
        if path.exists():
            return path.resolve()
    return "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_06_ab_startup/data/" + name


CSV_PATH = _find('startup_ab.csv')
df = pd.read_csv(CSV_PATH)
df['variant_b'] = (df['variant'] == 'B').astype(int)


## 0.1. Точечная оценка uplift

Перед bootstrap посчитайте estimand B−A. Интервал далее должен описывать неопределённость именно этой величины, а не отдельных конверсий.

**Эталон.** Эта ячейка фиксирует тот же контракт, который ученик заполняет до основного расчёта.

In [ ]:
point_uplift = float(
    df.loc[df['variant'] == 'B', 'converted'].mean()
    - df.loc[df['variant'] == 'A', 'converted'].mean()
)
assert -0.2 < point_uplift < 0.2

## 0.2. Контракт bootstrap

Зафиксируйте единицу ресемплирования, число повторов и квантили до расчёта. Группы A и B ресемплируются раздельно и с возвращением.

**Эталон.** Эта ячейка фиксирует тот же контракт, который ученик заполняет до основного расчёта.

In [ ]:
bootstrap_plan = {
    'unit': 'user', 'replace': True, 'n_iter': 2500,
    'quantiles': (0.025, 0.975), 'groups': 'resample separately',
}
assert bootstrap_plan['replace'] is True

## Решение 1

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
rng = np.random.default_rng(380)

a = df[df['variant'] == 'A']['converted'].to_numpy()

b = df[df['variant'] == 'B']['converted'].to_numpy()

## Решение 2

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
boot_diffs = []

for _ in range(2500):
    a_s = rng.choice(a, size=len(a), replace=True)
    b_s = rng.choice(b, size=len(b), replace=True)
    boot_diffs.append(float(b_s.mean() - a_s.mean()))

## Решение 3

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
ci_low, ci_high = np.quantile(boot_diffs, [0.025, 0.975])

CI_NOTE = (
    '95% CI — диапазон правдоподобных значений эффекта при выбранной процедуре. '
    'Если ноль вне интервала, эффект статистически совместим с отличием от нуля.'
)

corr_pages = float(df['pages_viewed'].corr(df['converted']))

## Решение 4

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
corr_time = float(df['session_minutes'].corr(df['converted']))

CAUSE_NOTE = (
    'Высокая корреляция не доказывает причинность: на обе переменные может влиять скрытый фактор '
    '(например, качество трафика или намерение пользователя купить).' 
)

b_conv = b

## Решение 5

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
boot_b = [float(rng.choice(b_conv, size=len(b_conv), replace=True).mean()) for _ in range(2500)]

ci_b = tuple(np.quantile(boot_b, [0.025, 0.975]))

rev = df['order_value'].to_numpy()

## Решение 6

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
boot_rev = [float(rng.choice(rev, size=len(rev), replace=True).mean()) for _ in range(2500)]

ci_rev = tuple(np.quantile(boot_rev, [0.025, 0.975]))

rows = []

## Решение 7

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
for dev in ['desktop', 'mobile']:
    part = df[df['device'] == dev]
    rows.append({'device': dev, 'corr_pages_conv': float(part['pages_viewed'].corr(part['converted']))})

## Решение 8

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
corr_table = pd.DataFrame(rows)

LIMIT_NOTE = (
    'Даже при CI и корреляциях остаются ограничения: синтетические данные, возможные скрытые факторы, '
    'и линейная связь может быть только приближением.'
)

print((round(ci_low, 4), round(ci_high, 4)), round(corr_pages, 4), round(corr_time, 4))

## Проверка преподавателя

Запустите `Run All`. Эталон должен завершиться без исключений; итоговые числа должны совпадать при повторном запуске благодаря фиксированным seed. Текстовый вывод проверяется на согласованность с направлением uplift, p-value и границами CI.